## Cell 1: Install Dependencies

Install all required packages for QLoRA fine-tuning. Run this cell first every time you start a new Kaggle session. `bitsandbytes` is required for 4-bit NF4 quantization.

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets wandb

## Cell 2: Configuration

**Edit this cell before each run.** Change RUN_NAME, LEARNING_RATE, and LORA_RANK to match the sweep plan in Cell 10.

- `RUN_NAME`: descriptive name logged to W&B — change this every run
- `LEARNING_RATE`: sweep values are 1e-4, 2e-4, 5e-4
- `LORA_RANK`: sweep values are 8, 16, 32
- `MAX_STEPS`: set to 500 for sweep runs, -1 for the final 3-epoch run
- `BASE_MODEL_REVISION`: **paste the exact commit hash from HF Hub before running** (go to https://huggingface.co/Qwen/Qwen2.5-7B-Instruct, click "Files and versions", copy the full commit hash)
- `RESUME_FROM_CHECKPOINT`: set to a checkpoint path like "/kaggle/working/checkpoints/checkpoint-400" to resume after a Kaggle session timeout; leave as None for a fresh start

In [ ]:
RUN_NAME = "qlora_lr_1e4_r16"  # descriptive name, change each run
LEARNING_RATE = 2e-4            # sweep: 1e-4, 2e-4, 5e-4
LORA_RANK = 16                  # sweep: 8, 16, 32
LORA_ALPHA = LORA_RANK * 2
MAX_STEPS = 500                 # sweep runs; set -1 for final run
NUM_EPOCHS = 3                  # only used when MAX_STEPS == -1
WANDB_PROJECT = "harmony-p3-qlora"
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
BASE_MODEL_REVISION = "FILL_BEFORE_RUNNING"  # paste commit hash from HF Hub
RESUME_FROM_CHECKPOINT = None   # set to checkpoint path to resume, e.g. "/kaggle/working/checkpoints/checkpoint-400"
SEED = 42

## Cell 3: Pre-flight Checks

**Run this before anything expensive.** Verifies GPU availability, data files, row counts, and sample format. If any assert fails, fix the issue before proceeding.

In [ ]:
import os, json, torch
from pathlib import Path

# 1. Verify GPU
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "No GPU detected — this notebook requires a T4×2 Kaggle environment"

# 2. Verify data files exist
TRAIN_PATH = "/kaggle/input/harmony-ade-data/train.jsonl"   # adjust dataset name if needed
VAL_PATH   = "/kaggle/input/harmony-ade-data/val.jsonl"
assert Path(TRAIN_PATH).exists(), f"MISSING: {TRAIN_PATH} — attach the Kaggle dataset before running"
assert Path(VAL_PATH).exists(),   f"MISSING: {VAL_PATH} — attach the Kaggle dataset before running"

# 3. Count rows
train_count = sum(1 for _ in open(TRAIN_PATH))
val_count   = sum(1 for _ in open(VAL_PATH))
print(f"Train rows: {train_count:,}  Val rows: {val_count:,}")
assert train_count > 15000, f"Expected ~19040 train rows, got {train_count}"
assert val_count   > 2000,  f"Expected ~2379 val rows, got {val_count}"

# 4. Spot-check first training example
with open(TRAIN_PATH) as f:
    sample = json.loads(f.readline())
print("\nSample user turn (first 200 chars):")
print(sample["messages"][0]["content"][:200])
print("\nSample assistant turn:")
print(sample["messages"][1]["content"])
assert "schema_version" in sample["messages"][1]["content"], "Assistant turn missing schema_version — check JSONL format"
print("\nPre-flight checks passed ✓")

## Cell 4: Load Tokenizer and Model (4-bit NF4)

Loads Qwen2.5-7B-Instruct in 4-bit NF4 quantization for QLoRA fine-tuning.

Key points:
- `bnb_4bit_compute_dtype=torch.float16` — NOT bfloat16. T4 is Turing architecture (compute capability 7.5); BF16 hardware was introduced in Ampere (8.0). Using float16 avoids slow software emulation.
- `bnb_4bit_use_double_quant=True` — double quantization: quantize the quantization constants themselves, saves ~0.4 bits/param.
- `prepare_model_for_kbit_training()` — **required** for QLoRA: enables gradient checkpointing for the 4-bit base model and casts layer norms to FP32 for training stability.
- `use_cache=False` — required for gradient checkpointing.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # float16, NOT bfloat16 — T4 Turing has no native BF16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for training

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False   # required for gradient checkpointing

model = prepare_model_for_kbit_training(model)   # REQUIRED for QLoRA: enables grad checkpointing + FP32 layer norms
print(f"Model loaded in 4-bit. dtype={model.dtype}")
print(f"Approx params: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

## Cell 5: LoRA Configuration

Configures the LoRA adapter for QLoRA. Same architecture as the FP16 LoRA notebook — `target_modules` covers all 7 linear projection layers. The base model is 4-bit; the adapter itself is trained in FP16.

In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
print(f"LoRA config: r={LORA_RANK}, alpha={LORA_ALPHA}")

## Cell 6: Load and Format Dataset

Loads the JSONL files and applies Qwen2.5's chat template to produce training strings. Also checks the token length distribution — ade_corpus_v2 sentences are short (p95 ~33 tokens), so 256 max_seq_length is safe headroom.

In [ ]:
import json
import numpy as np
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line.strip()))
    return rows

def format_example(example):
    """Apply Qwen2.5 chat template to produce the training string."""
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

raw_train = load_jsonl(TRAIN_PATH)
raw_val   = load_jsonl(VAL_PATH)

train_dataset = Dataset.from_list(raw_train).map(format_example)
val_dataset   = Dataset.from_list(raw_val).map(format_example)

print(f"Train: {len(train_dataset):,} examples")
print(f"Val:   {len(val_dataset):,} examples")

# Verify token length distribution
sample_lengths = [
    len(tokenizer(ex["text"], add_special_tokens=False)["input_ids"])
    for ex in train_dataset.select(range(200))
]
print(f"Token length p50={np.percentile(sample_lengths, 50):.0f}  p95={np.percentile(sample_lengths, 95):.0f}  max={max(sample_lengths)}")
assert max(sample_lengths) <= 512, f"Some examples exceed 512 tokens: {max(sample_lengths)} — investigate before training"

## Cell 7: SFT Training

Trains the model using SFTTrainer with loss masking (only assistant tokens contribute to loss). W&B is initialized before training so all metrics are logged even if training crashes mid-way.

**QLoRA-specific settings vs the LoRA notebook:**
- `optim="paged_adamw_8bit"` — pages optimizer states to CPU when VRAM is under pressure; required for QLoRA
- `fp16=False, bf16=False` — base model is already 4-bit; compute dtype is set in BitsAndBytesConfig
- All other settings are identical to the LoRA notebook for direct comparison

**OOM fallback:** If you get CUDA OOM (unlikely with QLoRA), set `per_device_train_batch_size=2` and `gradient_accumulation_steps=8` to keep effective batch size at 16.

In [ ]:
import wandb
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

wandb.init(project=WANDB_PROJECT, name=RUN_NAME, config={
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "max_steps": MAX_STEPS,
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "quantization": "4-bit NF4 double-quant",
})

# Loss masking: only compute loss on assistant tokens
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

sft_config = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    # --- Core training ---
    num_train_epochs=NUM_EPOCHS if MAX_STEPS == -1 else 100,  # large epoch count when using max_steps
    max_steps=MAX_STEPS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch = 16
    # If OOM: set per_device_train_batch_size=2 and gradient_accumulation_steps=8
    # --- Optimizer ---
    optim="paged_adamw_8bit",         # paged optimizer for QLoRA — pages optimizer states to CPU
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    # --- Precision ---
    fp16=False,                       # base model is 4-bit; compute dtype set in BitsAndBytesConfig
    bf16=False,
    # --- Memory ---
    gradient_checkpointing=True,
    # --- Sequence length ---
    max_seq_length=256,               # ade_corpus_v2 p95 ≈ 33 tokens; 256 is safe headroom
    dataset_text_field="text",
    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=100,
    # --- Checkpointing (CRITICAL for resumability) ---
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,               # keep last 3 checkpoints — enough to recover from timeout
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # --- Logging ---
    logging_steps=10,
    report_to="wandb",
    run_name=RUN_NAME,
    # --- Reproducibility ---
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    peft_config=lora_config,
)

print("Starting QLoRA training...")
print(f"  Run: {RUN_NAME}")
print(f"  LR={LEARNING_RATE}, rank={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"  MAX_STEPS={MAX_STEPS}  (use -1 for full 3-epoch final run)")
if RESUME_FROM_CHECKPOINT:
    print(f"  Resuming from: {RESUME_FROM_CHECKPOINT}")

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

print("\nTraining complete.")
print(f"Best eval loss: {trainer.state.best_metric}")

## Cell 8: Save Adapter + Tokenizer + Training Args

Saves the QLoRA adapter weights, tokenizer, and a training_args.json record. **Download /kaggle/working/adapter/ as a zip before the session ends.** Fill in `wandb_run_url` from W&B after training, and update `final_lr`/`final_rank`/`final_alpha` after the sweep is complete.

In [ ]:
import json
from pathlib import Path

ADAPTER_DIR = Path("/kaggle/working/adapter")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

training_args_data = {
    "method": "QLoRA",
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "run_name": RUN_NAME,
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    "optimizer": "paged_adamw_8bit",
    "quantization": "4-bit NF4 double-quant, compute_dtype=float16",
    "fp16": False,
    "bf16": False,
    "max_steps": MAX_STEPS,
    "num_epochs": NUM_EPOCHS,
    "effective_batch_size": 16,
    "max_seq_length": 256,
    "best_eval_loss": trainer.state.best_metric,
    "wandb_run_url": "FILL_AFTER_RUN",   # paste from W&B after training
    "kaggle_account": "team_member_2",
    "final_lr": "FILL_AFTER_SWEEP",
    "final_rank": "FILL_AFTER_SWEEP",
    "final_alpha": "FILL_AFTER_SWEEP",
}
with open(ADAPTER_DIR / "training_args.json", "w") as f:
    json.dump(training_args_data, f, indent=2)

print(f"Adapter saved to {ADAPTER_DIR}")
print("Files in adapter dir:")
for p in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

## Cell 9: Quick Inference Test

Verifies the QLoRA adapter generates output before you download it. Tests 3 clinical sentences. Valid JSON on all 3 is ideal; for sweep runs it is acceptable if some outputs are not valid JSON — check the final run carefully.

**IMPORTANT: Download /kaggle/working/adapter/ before the session ends.**

In [ ]:
import json as _json

INSTRUCTION_TEMPLATE = """You are a clinical information extractor. Given a clinical text, extract all
medications and adverse events as a JSON object that follows the schema below.
Return ONLY valid JSON. If no entity is present, return entities=[] and
relation_status=\"none\".

Return ONLY this JSON structure (no record_id, no validation block — those are added by the system):
{{
  \"schema_version\": \"v1\",
  \"entities\": [
    {{
      \"entity_type\": \"medication\" | \"adverse_event\",
      \"mention\": \"<string>\",
      \"dosage\": \"<string>\" | null,
      \"linked_medication\": \"<string>\" | null,
      \"evidence\": \"<string>\",
      \"source_span\": {{\"start_char\": <int>, \"end_char\": <int>}}
    }}
  ],
  \"relation_status\": \"related\" | \"not_related\" | \"none\"
}}

Clinical text:
{text}"""

TEST_SENTENCES = [
    "Intravenous azithromycin-induced ototoxicity.",
    "The patient had no adverse reactions to amoxicillin.",
    "She was started on metformin 500 mg twice daily.",
]

model.eval()
for text in TEST_SENTENCES:
    messages = [{"role": "user", "content": INSTRUCTION_TEMPLATE.format(text=text)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nInput: {text}")
    print(f"Output: {raw[:300]}")
    try:
        parsed = _json.loads(raw)
        print("✓ Valid JSON")
    except Exception:
        print("✗ Not valid JSON — acceptable for sweep runs, check final run")

print("\nInference test complete.")
print("⚠️  Download /kaggle/working/adapter/ before session ends.")

## Cell 10: 7-Run Sweep Plan

Follow this plan in order. Each sweep run takes ~10–15 minutes on T4×2 at MAX_STEPS=500. Compare eval_loss in W&B after each group before moving to the next. Results from this notebook (QLoRA) will be directly compared with the LoRA-FP16 notebook (p3_02_lora_train.ipynb) to measure the effect of 4-bit quantization on F1.

### 7-Run Sweep Plan

**Phase 1 — Learning Rate Sweep** (fix rank=16, sweep LR)

| Run | RUN_NAME | LEARNING_RATE | LORA_RANK | MAX_STEPS |
|-----|----------|---------------|-----------|-----------|
| 1 | `qlora_lr_1e4_r16` | 1e-4 | 16 | 500 |
| 2 | `qlora_lr_2e4_r16` | 2e-4 | 16 | 500 |
| 3 | `qlora_lr_5e4_r16` | 5e-4 | 16 | 500 |

→ Compare `eval_loss` in W&B. Pick the **best LR** (lowest eval_loss).

---

**Phase 2 — Rank Sweep** (fix best LR, sweep rank)

| Run | RUN_NAME | LEARNING_RATE | LORA_RANK | MAX_STEPS |
|-----|----------|---------------|-----------|-----------|
| 4 | `qlora_bestlr_r8` | best_LR | 8 | 500 |
| 5 | `qlora_bestlr_r16` | best_LR | 16 | 500 |
| 6 | `qlora_bestlr_r32` | best_LR | 32 | 500 |

→ Compare `eval_loss` in W&B. Pick the **best rank** (lowest eval_loss).

---

**Final Run** (full 3-epoch training)

| Run | RUN_NAME | LEARNING_RATE | LORA_RANK | MAX_STEPS | NUM_EPOCHS |
|-----|----------|---------------|-----------|-----------|------------|
| 7 | `qlora_final` | best_LR | best_rank | -1 | 3 |

→ This is the **production QLoRA adapter**. After Run 7:
1. Fill `wandb_run_url` in Cell 8 with the W&B run URL
2. Update `final_lr`, `final_rank`, `final_alpha` in Cell 8
3. Download `/kaggle/working/adapter/` as a zip
4. Commit to `models/adapters/qlora_v1/`

---

**Comparing LoRA vs QLoRA results:**

After both Run 7s are complete, compare the two adapters:
- If QLoRA eval_loss ≈ LoRA eval_loss (within 5%): quantization does not hurt, use QLoRA for production (60% lower VRAM, faster inference)
- If QLoRA eval_loss is notably higher: quantization noise is measurable, prefer LoRA adapter for production